# EDA – Solar Eclipse Dataset

Kort utforskning av NASA:s katalog över solförmörkelser, som ligger till grund för eClipseBord.

Syftet är en enklare undersökning av datan för att kunna använda den i Azure-deployen.

**Datakälla:** `data/solar.csv`

## Load the data

Reading everything as strings first, so pandas doesn't guess the types for me.
Want to see what the data actually looks like before converting anything.

In [9]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("..") / "data" / "solar.csv"

df = pd.read_csv(DATA_PATH, dtype=str)
df.shape

(11898, 15)

## Columns and missing values

Checking what columns exist and where the gaps are.

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11898 entries, 0 to 11897
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Catalog Number     11898 non-null  str  
 1   Calendar Date      11898 non-null  str  
 2   Eclipse Time       11898 non-null  str  
 3   Delta T (s)        11898 non-null  str  
 4   Lunation Number    11898 non-null  str  
 5   Saros Number       11898 non-null  str  
 6   Eclipse Type       11898 non-null  str  
 7   Gamma              11898 non-null  str  
 8   Eclipse Magnitude  11898 non-null  str  
 9   Latitude           11898 non-null  str  
 10  Longitude          11898 non-null  str  
 11  Sun Altitude       11898 non-null  str  
 12  Sun Azimuth        11898 non-null  str  
 13  Path Width (km)    7698 non-null   str  
 14  Central Duration   7698 non-null   str  
dtypes: str(15)
memory usage: 2.2 MB


Note: 13 of 15 columns are complete. `Path Width` and `Central Duration` are both
missing in 4200 rows. Probably because partial eclipses have no central line —
checking that below.

## Eclipse types

Looking at the type codes to see what values I can filter on later.

In [11]:
df["Eclipse Type"].value_counts()

Eclipse Type
P     3875
A     3755
T     3049
H      502
Pb     163
Pe     162
Tm      72
Am      72
An      36
A+      34
A-      34
H3      26
As      25
H2      24
Hm      17
T-      17
Tn      14
Ts      12
T+       9
Name: count, dtype: int64

19 different codes, but only 4 main types (P, A, T, H). The suffixes are NASA's
notation for special cases, e.g. eclipses at sunrise or sunset.
Too many categories for a dropdown filter, so grouping on the first letter instead.

In [12]:
df["Eclipse Type"].str[0].value_counts()

Eclipse Type
P    4200
A    3956
T    3173
H     569
Name: count, dtype: int64

4 clean groups. P = 4200, which matches the missing values above exactly —
so partial eclipses really are the ones without a central line

## Extracting the year

The date column has years before Christ as negative numbers (e.g. "-1999 June 12"),
which pandas can't parse as a real date. Taking the year as an integer instead —
month and day don't matter when the range is 5000 years.

In [13]:
df["year"] = df["Calendar Date"].str.split(" ").str[0].astype(int)
df["year"].describe()

count    11898.000000
mean       499.962431
std       1447.767195
min      -1999.000000
25%       -745.750000
50%        504.000000
75%       1754.000000
max       3000.000000
Name: year, dtype: float64

Range is -1999 to 3000, all 11898 rows converted without errors.
Median year is 504, so half the dataset is ancient history.
Confirms I need a year filter in the frontend, otherwise the modern
eclipses drown in the old data.

In [14]:
modern = df[df["year"] >= 1900]
len(modern)

2618

2618 rows from 1900 onwards — about a fifth of the dataset.
Good default range for the dashboard. Note that this includes future
eclipses up to year 3000, not just historical ones.

## Conclusion — what the backend needs

Columns to expose in the API:

| Column | Why |
|---|---|
| `year` | derived from Calendar Date, used for filtering and as x-axis |
| `Eclipse Type` | grouped to first letter (P/A/T/H), used as filter and category |
| `Calendar Date` | keep the original for the data table |
| `Eclipse Magnitude` | numeric, useful for a second chart |
| `Saros Number` | eclipse cycle, could be interesting to group by |

Cleaning needed before serving:
- extract `year` as int from `Calendar Date`
- take first letter of `Eclipse Type`
- convert `Eclipse Magnitude` to float

Skipping Latitude/Longitude — no map in this dashboard.
Skipping Path Width and Central Duration — missing for all partial eclipses.

Planned endpoints:
- `GET /health` — already built
- `GET /eclipses` — list, with optional year range and type filter
- `GET /eclipses/count-by-type` — aggregated counts for the chart